# Deschênes & Greenstone (2011)

Illinois county-level LOCA2 indicators. Keep this notebook and `loca2_common.py` in the same folder. Open Jupyter in that folder and run the cells from top to bottom. Each notebook uses its own kernel.

Existing files are skipped (`OVERWRITE = False`). County aggregation uses cosine-latitude area correction.

This workflow requires both relative-humidity inputs as well as temperature and precipitation.

Go to loca2_common.py to set your input and output path. 

Indicators calculated: No of days in a year with Mean Temperature in specific bins, Total Annual Precipitation in specific bins

## 1. Import shared functions

Required packages: numpy, xarray, dask, netCDF4, cftime, pandas, geopandas, regionmask, and tqdm. Use your existing Keeling environment.

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import loca2_common as common
from loca2_common import (
    aggregate_to_illinois_counties,
    annual_sum,
    annual_temperature_bin_counts,
    daily_mean_temperature,
    to_fahrenheit,
)

## 2. Paper settings

In [2]:
PAPER_NAME = 'Deschênes & Greenstone (2011)'
REQUIRED_VARIABLES = ('tasmin', 'tasmax', 'pr')
OVERWRITE = False

## 3. Check paths and available runs

This cell lists the workload and loads county boundaries. It does not write climate output.

In [3]:
output_directory = common.PAPER_OUTPUT_ROOT / common.paper_directory_name(PAPER_NAME)
jobs = common.discover_paper_runs(REQUIRED_VARIABLES)
existing = sum(
    (output_directory / (
        f"{common.safe_token(PAPER_NAME)}_"
        f"{common.safe_token(model)}_{common.safe_token(member)}_{scenario}.nc"
    )).exists()
    for model, member, scenario in jobs
)
print("Input root:", common.PAPER_INPUT_ROOT)
print("Output folder:", output_directory)
print("Eligible runs:", len(jobs))
print("Existing files to skip:", existing if not OVERWRITE else 0)
print("Runs to process:", len(jobs) - existing if not OVERWRITE else len(jobs))
if not jobs:
    raise FileNotFoundError("No eligible runs found. Check input paths and required variables.")
counties = common.load_counties()
print("Counties:", len(counties))

Input root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois
Output folder: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/temp/Deschênes & Greenstone (2011)
Eligible runs: 156
Existing files to skip: 0
Runs to process: 156
Counties: 102


## 4. Paper indicator calculation

In [4]:
def calculate_deschenes_greenstone(inputs):
    tmean_c = daily_mean_temperature(inputs['tasmin'], inputs['tasmax'])
    tmean_f = to_fahrenheit(tmean_c)
    edges_f = [-np.inf, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, np.inf]
    labels = ['tmean_days_lt10F', 'tmean_days_10_20F', 'tmean_days_20_30F', 'tmean_days_30_40F', 'tmean_days_40_50F', 'tmean_days_50_60F', 'tmean_days_60_70F', 'tmean_days_70_80F', 'tmean_days_80_90F', 'tmean_days_ge90F']
    variables = annual_temperature_bin_counts(temperature=tmean_f, edges=edges_f, labels=labels)
    variables['annual_precipitation'] = annual_sum(inputs['pr']).rename('annual_precipitation')
    result = xr.Dataset(variables)
    for variable in labels:
        result[variable].attrs.update(units='days', frequency='annual', temperature_statistic='daily mean temperature calculated as (tasmin + tasmax) / 2')
    result['annual_precipitation'].attrs.update(units='inch', long_name='annual total precipitation', temporal_aggregation='sum of daily precipitation')
    result.attrs.update(reference_temperature_bin='50 to less than 60 degrees Fahrenheit')
    result = aggregate_to_illinois_counties(dataset=result, counties=counties)
    annual_precipitation = result['annual_precipitation']
    precip_conditions = {'precip_category_lt10in': annual_precipitation < 10.0, 'precip_category_10_15in': (annual_precipitation >= 10.0) & (annual_precipitation < 15.0), 'precip_category_15_20in': (annual_precipitation >= 15.0) & (annual_precipitation < 20.0), 'precip_category_20_25in': (annual_precipitation >= 20.0) & (annual_precipitation < 25.0), 'precip_category_25_30in': (annual_precipitation >= 25.0) & (annual_precipitation < 30.0), 'precip_category_30_35in': (annual_precipitation >= 30.0) & (annual_precipitation < 35.0), 'precip_category_35_40in': (annual_precipitation >= 35.0) & (annual_precipitation < 40.0), 'precip_category_40_45in': (annual_precipitation >= 40.0) & (annual_precipitation < 45.0), 'precip_category_45_50in': (annual_precipitation >= 45.0) & (annual_precipitation < 50.0), 'precip_category_50_55in': (annual_precipitation >= 50.0) & (annual_precipitation < 55.0), 'precip_category_55_60in': (annual_precipitation >= 55.0) & (annual_precipitation <= 60.0), 'precip_category_gt60in': annual_precipitation > 60.0}
    for variable_name, condition in precip_conditions.items():
        category = xr.where(annual_precipitation.notnull(), condition.astype('int8'), np.nan)
        category.attrs.update(units='1', frequency='annual', coding='1 if county-average annual precipitation is within this category; otherwise 0', precipitation_statistic='county-average annual total precipitation')
        result[variable_name] = category
    result.attrs.update(spatial_resolution='Illinois county', precipitation_category_assignment='categories assigned after county aggregation using county-average annual precipitation')
    return result

## 5. Generate output

Existing matching filenames are skipped. A skipped file is not recalculated or checked for completeness.

In [ ]:
common.run_paper(
    paper_name=PAPER_NAME,
    required_variables=REQUIRED_VARIABLES,
    calculation_function=calculate_deschenes_greenstone,
    overwrite=OVERWRITE,
)

## 6. Inspect an output file

In [ ]:
files = sorted(output_directory.glob(f"{common.safe_token(PAPER_NAME)}_*.nc"))
if files:
    with xr.open_dataset(files[0], decode_timedelta=False) as dataset:
        print("Example:", files[0].name)
        print("Dimensions:", dict(dataset.sizes))
        print("Variables:", list(dataset.data_vars))
        print("Spatial resolution:", dataset.attrs.get("spatial_resolution"))
        assert dataset.sizes.get("county") == 102, "Output is not a 102-county dataset."
else:
    print("No output files found. Review the processing messages above.")